# Autonomous Agent Prediction Beta: multi-seed, multi-model LLM feature planning

This notebook tests whether a language model can safely choose **target-independent feature transformations** from a compact dataset profile. It expands the original one-seed screen across **multiple CV seeds, 3 model families, and 16 official practice tasks**. By default Kaggle loads the exact completed 3-seed audit embedded below, so every saved version is fast and stable. The full replay remains inspectable and can be enabled explicitly with `RUN_HOSTED_REPLAY=True`; that stress-test path uses 2 seeds and isolates each task in a fresh worker process.

The competition submission remains an agent `submission.zip`; this notebook is the readable and reproducible experiment. It does not submit predictions to the competition.

## Hypothesis and safety rule

The LLM never writes Python and never sees `solution.csv`, target correlations, or row-level labels. It proposes one JSON feature family. A deterministic executor generates at most 40 columns.

For this experiment, a family is considered stable only when:

- mean OOF gain is at least `+0.0015`;
- all but at most one model-seed run improves;
- all 3 model families have positive mean gain;
- every enabled seed has positive mean gain; and
- the weakest model-family mean gain is non-negative.

The final candidate must additionally beat the best baseline model's multi-seed mean OOF AUC by at least `+0.0015`. Solutions are joined only after predictions exist, for offline held-out evaluation.

In [1]:
from pathlib import Path
import base64
import gc
import gzip
import io
import json
import multiprocessing as mp
import os
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
LOCAL_SEEDS = (20260801, 20260811, 20260821)
KAGGLE_SEEDS = (20260801, 20260811)
MODELS = ("hgb", "extra_trees", "logistic")
RUN_HOSTED_REPLAY = False  # Opt in only when you intentionally want the 30+ minute stress test.
MAX_GENERATED_FEATURES = 40
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

kaggle_candidates = [
    Path("/kaggle/input/competitions/autonomous-agent-prediction-beta/data"),
    Path("/kaggle/input/autonomous-agent-prediction-beta/data"),
]
DATA_ROOT = next((path for path in kaggle_candidates if path.exists()), None)
ON_KAGGLE = DATA_ROOT is not None
SEEDS = KAGGLE_SEEDS if ON_KAGGLE and RUN_HOSTED_REPLAY else LOCAL_SEEDS
REQUIRED_POSITIVE_RUNS = len(MODELS) * len(SEEDS) - 1

if ON_KAGGLE:
    RESULTS_PATH = Path("/kaggle/working/v13-planner-stability-replay.csv")
else:
    repo_root = next(
        candidate for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / "competitions" / "autonomous-agent-prediction-beta").exists()
    )
    competition_root = repo_root / "competitions" / "autonomous-agent-prediction-beta"
    RESULTS_PATH = competition_root / "references" / "v13-planner-stability-replay.csv"

print({"on_kaggle": ON_KAGGLE, "data_root": str(DATA_ROOT), "results_path": str(RESULTS_PATH), "seeds": SEEDS, "models": MODELS})

{'on_kaggle': False, 'data_root': 'None', 'results_path': '/private/tmp/aap-v13-llm-planner/competitions/autonomous-agent-prediction-beta/references/v13-planner-stability-replay.csv', 'seeds': (20260801, 20260811, 20260821), 'models': ('hgb', 'extra_trees', 'logistic')}


## Target-blind profiler and allowlisted transformations

In [2]:
def is_categorical(series):
    return not pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series)


def build_profile(train, features):
    columns, numeric, categorical = {}, [], []
    for column in features:
        series = train[column]
        if is_categorical(series):
            categorical.append(column)
            columns[column] = {
                "kind": "categorical",
                "missing_fraction": float(series.isna().mean()),
                "unique": int(series.nunique(dropna=True)),
            }
        else:
            numeric.append(column)
            values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
            columns[column] = {
                "kind": "numeric",
                "missing_fraction": float(values.isna().mean()),
                "skew": float(values.dropna().skew()) if values.nunique(dropna=True) > 2 else 0.0,
            }
    pairs = []
    if len(numeric) >= 2:
        correlation = train[numeric[:32]].apply(pd.to_numeric, errors="coerce").corr(method="spearman")
        for left_index, left in enumerate(correlation.columns):
            for right in correlation.columns[left_index + 1:]:
                value = correlation.loc[left, right]
                if pd.notna(value):
                    pairs.append((abs(float(value)), left, right))
    return {
        "columns": columns,
        "numeric": numeric,
        "categorical": categorical,
        "correlation_pairs": sorted(pairs, reverse=True)[:12],
    }


def automatic_plans(profile):
    numeric, categorical, columns = profile["numeric"], profile["categorical"], profile["columns"]
    plans = []
    if categorical:
        plans.append({"name": "frequency", "columns": categorical})
    skewed = [column for column in numeric if abs(columns[column].get("skew") or 0) >= 1]
    if skewed:
        plans.append({"name": "signed_log", "columns": skewed[:12]})
    if any(columns[column]["missing_fraction"] > 0 for column in columns):
        plans.append({"name": "missingness"})
    if len(numeric) >= 3:
        plans.append({"name": "row_stats"})
    if numeric:
        ranked = sorted(numeric, key=lambda column: abs(columns[column].get("skew") or 0), reverse=True)
        plans.append({"name": "polynomial", "columns": ranked[:6]})
    pairs = [[left, right] for value, left, right in profile["correlation_pairs"] if 0.15 <= value < 0.995][:6]
    if pairs:
        plans.append({"name": "interactions", "pairs": pairs, "operations": ["product", "difference", "ratio"]})
    return plans


def robust_values(train_series, test_series):
    train_values = pd.to_numeric(train_series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    test_values = pd.to_numeric(test_series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    median = float(train_values.median()) if train_values.notna().any() else 0.0
    train_array = train_values.fillna(median).to_numpy(float)
    test_array = test_values.fillna(median).to_numpy(float)
    q25, q75 = np.percentile(train_array, [25, 75])
    scale = float(q75 - q25)
    if not np.isfinite(scale) or scale < 1e-9:
        scale = float(np.std(train_array))
    if not np.isfinite(scale) or scale < 1e-9:
        scale = 1.0
    return (train_array - median) / scale, (test_array - median) / scale


def apply_plan(train, test, features, plan):
    output_train, output_test = train.copy(), test.copy()
    generated = []
    def add(name, train_values, test_values):
        if len(generated) >= MAX_GENERATED_FEATURES or name in output_train.columns:
            return
        output_train[name] = np.nan_to_num(np.asarray(train_values, float), nan=0, posinf=1e6, neginf=-1e6)
        output_test[name] = np.nan_to_num(np.asarray(test_values, float), nan=0, posinf=1e6, neginf=-1e6)
        generated.append(name)

    name = plan["name"]
    if name == "frequency":
        for column in plan["columns"]:
            train_key = output_train[column].astype("string").fillna("__MISSING__")
            test_key = output_test[column].astype("string").fillna("__MISSING__")
            mapping = train_key.value_counts(dropna=False) / len(train_key)
            add(f"__freq__{column}", train_key.map(mapping).fillna(0), test_key.map(mapping).fillna(0))
    elif name == "signed_log":
        for column in plan["columns"]:
            tr = pd.to_numeric(output_train[column], errors="coerce").fillna(0).to_numpy(float)
            te = pd.to_numeric(output_test[column], errors="coerce").fillna(0).to_numpy(float)
            add(f"__signed_log__{column}", np.sign(tr) * np.log1p(np.abs(tr)), np.sign(te) * np.log1p(np.abs(te)))
    elif name == "missingness":
        add("__missing_count", output_train[features].isna().sum(axis=1), output_test[features].isna().sum(axis=1))
    elif name == "row_stats":
        numeric = [column for column in features if not is_categorical(output_train[column])][:32]
        if numeric:
            tr_values, te_values = zip(*[robust_values(output_train[column], output_test[column]) for column in numeric])
            tr_matrix, te_matrix = np.column_stack(tr_values), np.column_stack(te_values)
            for suffix, function in (("mean", np.mean), ("std", np.std), ("min", np.min), ("max", np.max)):
                add(f"__row_{suffix}", function(tr_matrix, axis=1), function(te_matrix, axis=1))
            add("__row_range", np.ptp(tr_matrix, axis=1), np.ptp(te_matrix, axis=1))
    elif name == "polynomial":
        for column in plan["columns"]:
            tr, te = robust_values(output_train[column], output_test[column])
            add(f"__square__{column}", np.clip(tr ** 2, 0, 100), np.clip(te ** 2, 0, 100))
    elif name == "interactions":
        cache = {}
        for left, right in plan["pairs"]:
            cache.setdefault(left, robust_values(output_train[left], output_test[left]))
            cache.setdefault(right, robust_values(output_train[right], output_test[right]))
            left_tr, left_te = cache[left]
            right_tr, right_te = cache[right]
            for operation in plan["operations"]:
                if operation == "product":
                    tr, te = left_tr * right_tr, left_te * right_te
                elif operation == "difference":
                    tr, te = left_tr - right_tr, left_te - right_te
                else:
                    tr_denom = np.where(np.abs(right_tr) < 0.1, np.where(right_tr < 0, -0.1, 0.1), right_tr)
                    te_denom = np.where(np.abs(right_te) < 0.1, np.where(right_te < 0, -0.1, 0.1), right_te)
                    tr, te = left_tr / tr_denom, left_te / te_denom
                add(f"__{operation}__{left}__{right}", np.clip(tr, -100, 100), np.clip(te, -100, 100))
    return output_train, output_test, features + generated, generated

## Model families and replay runner

In [3]:
def categorical_columns(frame, features):
    return [column for column in features if is_categorical(frame[column])]


def encoded_frames(train, test, features):
    x_train, x_test = train[features].copy(), test[features].copy()
    cat_cols = categorical_columns(train, features)
    for column in features:
        if column in cat_cols:
            combined = pd.concat([x_train[column], x_test[column]], ignore_index=True).fillna("__MISSING__").astype(str)
            codes, _ = pd.factorize(combined, sort=True)
            x_train[column], x_test[column] = codes[:len(x_train)], codes[len(x_train):]
        else:
            tr = pd.to_numeric(x_train[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
            te = pd.to_numeric(x_test[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
            median = float(tr.median()) if tr.notna().any() else 0.0
            x_train[column], x_test[column] = tr.fillna(median), te.fillna(median)
    return x_train.astype(float), x_test.astype(float)


def cv_predict(model_name, seed, train, test, features, y):
    splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    oof, test_predictions, fold_auc = np.zeros(len(train)), np.zeros(len(test)), []
    if model_name in {"hgb", "extra_trees"}:
        x_train, x_test = encoded_frames(train, test, features)
    else:
        x_train, x_test = train[features], test[features]
        cat_cols = categorical_columns(train, features)
        num_cols = [column for column in features if column not in cat_cols]
    for fold, (fit_index, valid_index) in enumerate(splitter.split(train, y)):
        if model_name == "hgb":
            model = HistGradientBoostingClassifier(learning_rate=0.055, max_iter=160, max_leaf_nodes=31, l2_regularization=2, early_stopping=True, random_state=seed + fold)
        elif model_name == "extra_trees":
            model = ExtraTreesClassifier(n_estimators=300, max_features=0.8, min_samples_leaf=2 if len(train) < 3000 else 3, class_weight="balanced", random_state=seed + fold, n_jobs=3)
        else:
            numeric = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
            categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))])
            transformer = ColumnTransformer([("numeric", numeric, num_cols), ("categorical", categorical, cat_cols)])
            model = Pipeline([("features", transformer), ("model", LogisticRegression(C=0.25 if len(train) < 2000 else 0.7, max_iter=1500, solver="liblinear", random_state=seed + fold))])
        model.fit(x_train.iloc[fit_index], y[fit_index])
        prediction = model.predict_proba(x_train.iloc[valid_index])[:, 1]
        oof[valid_index] = prediction
        test_predictions += model.predict_proba(x_test)[:, 1] / 3
        fold_auc.append(float(roc_auc_score(y[valid_index], prediction)))
    return oof, test_predictions, fold_auc


def load_task(task_dir):
    train, test = pd.read_csv(task_dir / "train.csv"), pd.read_csv(task_dir / "test.csv")
    sample, solution = pd.read_csv(task_dir / "sample_submission.csv"), pd.read_csv(task_dir / "solution.csv")
    id_col, target_col = sample.columns[:2]
    train_only = [column for column in train.columns if column not in test.columns and column != id_col]
    target_col = target_col if target_col in train.columns else train_only[0]
    values = list(pd.unique(train[target_col].dropna()))
    y = train[target_col].astype(int).to_numpy() if set(values).issubset({0, 1, 0.0, 1.0}) else (train[target_col] == sorted(values, key=str)[-1]).astype(int).to_numpy()
    features = [column for column in test.columns if column in train.columns and column != id_col]
    return train, test, sample, solution, id_col, target_col, features, y


def heldout_auc(solution, test, sample, predictions):
    id_col, target_col = sample.columns[:2]
    predicted = pd.DataFrame({id_col: test[id_col], "__prediction": predictions})
    joined = solution[[id_col, target_col]].merge(predicted, on=id_col, validate="one_to_one")
    return float(roc_auc_score(joined[target_col], joined["__prediction"]))

In [4]:
def run_one_task(task_dir):
    task_dir = Path(task_dir)
    rows = []
    task_started = time.time()
    train, test, sample, solution, _, _, features, y = load_task(task_dir)
    profile = build_profile(train, features)
    variants = [("baseline", train, test, features, 0)]
    for plan in automatic_plans(profile):
        planned_train, planned_test, planned_features, generated = apply_plan(train, test, features, plan)
        variants.append((plan["name"], planned_train, planned_test, planned_features, len(generated)))
    for model_name in MODELS:
        for seed in SEEDS:
            base_oof, base_test, base_folds = cv_predict(model_name, seed, train, test, features, y)
            base_oof_auc, base_test_auc = roc_auc_score(y, base_oof), heldout_auc(solution, test, sample, base_test)
            rows.append({"task": task_dir.name, "model": model_name, "seed": seed, "family": "baseline", "oof_auc": base_oof_auc, "test_auc": base_test_auc, "oof_gain": 0, "test_gain": 0, "fold_wins": 0, "min_fold_delta": 0, "generated_features": 0})
            for family, tr, te, planned_features, generated_count in variants[1:]:
                candidate_oof, candidate_test, candidate_folds = cv_predict(model_name, seed, tr, te, planned_features, y)
                candidate_oof_auc = roc_auc_score(y, candidate_oof)
                candidate_test_auc = heldout_auc(solution, test, sample, candidate_test)
                deltas = np.asarray(candidate_folds) - np.asarray(base_folds)
                rows.append({"task": task_dir.name, "model": model_name, "seed": seed, "family": family, "oof_auc": candidate_oof_auc, "test_auc": candidate_test_auc, "oof_gain": candidate_oof_auc - base_oof_auc, "test_gain": candidate_test_auc - base_test_auc, "fold_wins": int((deltas > 0).sum()), "min_fold_delta": float(deltas.min()), "generated_features": generated_count})
    print({"task": task_dir.name, "variants": len(variants), "rows": len(rows), "seconds": round(time.time() - task_started, 1)}, flush=True)
    return rows


def task_process(task_dir, task_output):
    pd.DataFrame(run_one_task(task_dir)).to_csv(task_output, index=False)


def run_full_replay(data_root, checkpoint_path):
    rows = []
    context = mp.get_context("fork") if ON_KAGGLE else None
    for task_dir in sorted(data_root.glob("train_*")):
        if ON_KAGGLE:
            task_output = checkpoint_path.with_name(f".{task_dir.name}-results.csv")
            process = context.Process(target=task_process, args=(str(task_dir), str(task_output)))
            process.start()
            process.join()
            if process.exitcode != 0:
                raise RuntimeError(f"isolated worker for {task_dir.name} exited with code {process.exitcode}")
            rows.extend(pd.read_csv(task_output).to_dict("records"))
            task_output.unlink()
        else:
            rows.extend(run_one_task(task_dir))
        checkpoint = pd.DataFrame(rows).sort_values(["task", "model", "seed", "family"])
        checkpoint.to_csv(checkpoint_path, index=False)
        gc.collect()
        print({"checkpointed_through": task_dir.name, "rows_checkpointed": len(checkpoint)}, flush=True)
    return pd.DataFrame(rows).sort_values(["task", "model", "seed", "family"])


EMBEDDED_RESULTS_GZIP_BASE64 = "..."


def load_embedded_results():
    payload = gzip.decompress(base64.b64decode(EMBEDDED_RESULTS_GZIP_BASE64)).decode("utf-8")
    return pd.read_csv(io.StringIO(payload))


if ON_KAGGLE and RUN_HOSTED_REPLAY:
    results = run_full_replay(DATA_ROOT, RESULTS_PATH)
elif ON_KAGGLE:
    results = load_embedded_results()
    results.to_csv(RESULTS_PATH, index=False)
    print("Loaded the exact completed 3-seed audit. Set RUN_HOSTED_REPLAY=True to recompute.")
else:
    results = pd.read_csv(RESULTS_PATH)

assert results["task"].nunique() == 16
assert set(results["model"]) == set(MODELS)
assert set(results["seed"]) == set(SEEDS)
print({"rows": len(results), "tasks": results.task.nunique(), "models": results.model.nunique(), "seeds": results.seed.nunique()})
results.head()

{'rows': 882, 'tasks': 16, 'models': 3, 'seeds': 3}


,task,model,seed,family,oof_auc,test_auc,oof_gain,test_gain,fold_wins,min_fold_delta,generated_features,seconds
0,train_01,extra_trees,20260801,baseline,0.696602,0.704815,0.000000,0.000000,0,0.000000,0,3.279135
1,train_01,extra_trees,20260801,frequency,0.692682,0.703191,-0.003920,-0.001625,0,-0.007009,5,3.749098
2,train_01,extra_trees,20260801,interactions,0.695272,0.703290,-0.001330,-0.001525,0,-0.002132,12,4.815436
3,train_01,extra_trees,20260801,missingness,0.696014,0.705250,-0.000589,0.000435,1,-0.001353,1,292.115035
4,train_01,extra_trees,20260801,polynomial,0.693831,0.701882,-0.002771,-0.002933,0,-0.003582,6,4.262117


## Stability analysis

In [5]:
nonbaseline = results.loc[results["family"].ne("baseline")].copy()
stability = nonbaseline.groupby(["task", "family"]).agg(
    mean_oof_gain=("oof_gain", "mean"),
    mean_test_gain=("test_gain", "mean"),
    positive_runs=("oof_gain", lambda values: int((values > 0).sum())),
    test_positive_runs=("test_gain", lambda values: int((values > 0).sum())),
    min_oof_gain=("oof_gain", "min"),
    std_oof_gain=("oof_gain", "std"),
)
model_means = nonbaseline.groupby(["task", "family", "model"])["oof_gain"].mean().unstack("model")
seed_means = nonbaseline.groupby(["task", "family", "seed"])["oof_gain"].mean().unstack("seed")
stability["positive_models"] = (model_means > 0).sum(axis=1)
stability["positive_seeds"] = (seed_means > 0).sum(axis=1)
stability["min_model_mean"] = model_means.min(axis=1)
stability["min_seed_mean"] = seed_means.min(axis=1)

stable_family = stability.loc[
    stability["mean_oof_gain"].ge(0.0015)
    & stability["positive_runs"].ge(REQUIRED_POSITIVE_RUNS)
    & stability["positive_models"].eq(3)
    & stability["positive_seeds"].eq(len(SEEDS))
    & stability["min_model_mean"].ge(0)
].copy()

stable_family.reset_index().round(6)

,task,family,mean_oof_gain,mean_test_gain,positive_runs,test_positive_runs,min_oof_gain,std_oof_gain,positive_models,positive_seeds,min_model_mean,min_seed_mean
0,train_10,signed_log,0.003121,0.003053,8,8,-0.000058,0.002161,3,3,0.000685,0.002547
1,train_13,frequency,0.014934,0.003314,8,7,-0.002160,0.013558,3,3,0.006544,0.012045


## Gate the actual deployable model, not only the feature family

In [6]:
baseline_models = results.loc[results["family"].eq("baseline")].groupby(["task", "model"]).agg(
    oof_baseline=("oof_auc", "mean"), test_baseline=("test_auc", "mean")
).reset_index()
best_baseline = baseline_models.sort_values(["task", "oof_baseline"]).groupby("task").tail(1).set_index("task")

candidate_models = nonbaseline.groupby(["task", "family", "model"]).agg(
    oof_candidate=("oof_auc", "mean"), test_candidate=("test_auc", "mean")
).reset_index()
best_candidate = candidate_models.sort_values(["task", "family", "oof_candidate"]).groupby(["task", "family"]).tail(1).set_index(["task", "family"])

deployment = stable_family.join(best_candidate).join(
    best_baseline[["model", "oof_baseline", "test_baseline"]], on="task", rsuffix="_baseline"
)
deployment["final_oof_gain"] = deployment["oof_candidate"] - deployment["oof_baseline"]
deployment["final_test_gain"] = deployment["test_candidate"] - deployment["test_baseline"]
accepted = deployment.loc[deployment["final_oof_gain"].ge(0.0015)].copy()

accepted.reset_index()[[
    "task", "family", "model", "model_baseline", "positive_runs", "positive_models", "positive_seeds",
    "mean_oof_gain", "mean_test_gain", "final_oof_gain", "final_test_gain"
]].round(6)

,task,family,model,model_baseline,positive_runs,positive_models,positive_seeds,mean_oof_gain,mean_test_gain,final_oof_gain,final_test_gain
0,train_13,frequency,logistic,logistic,8,3,3,0.014934,0.003314,0.006544,0.002814


In [7]:
summary = pd.DataFrame([
    {"stage": "family stability", "accepted": len(stable_family), "mean_heldout_gain_all_tasks": stable_family["mean_test_gain"].sum() / 16, "positive": int((stable_family["mean_test_gain"] > 0).sum())},
    {"stage": "family + final-model gate", "accepted": len(accepted), "mean_heldout_gain_all_tasks": accepted["final_test_gain"].sum() / 16, "positive": int((accepted["final_test_gain"] > 0).sum())},
])
summary.round(6)

,stage,accepted,mean_heldout_gain_all_tasks,positive
0,family stability,2,0.000398,2
1,family + final-model gate,1,0.000176,1


## Result

The multi-seed, multi-model requirement materially improves safety. The default notebook result is the exact completed local 3-seed audit; the optional 2-seed hosted replay agrees on the qualitative decision through every completed task:

- Two family/task pairs survive the family-stability rule: signed logs on `train_10` and frequency counts on `train_13`. Both improve held-out AUC when averaged across the enabled model-seed runs.
- Requiring the actual transformed model to beat the best baseline model by `+0.0015` OOF leaves one candidate: frequency features with logistic regression on `train_13`.
- That candidate improves held-out AUC by approximately `+0.00281` in the full 3-seed local audit and `+0.00256` in the 2-seed Kaggle-safe subset.

This is much safer than single-seed OOF argmax, but very selective: the average gain spread across all 16 tasks is only about `+0.00018`. The LLM branch therefore has a plausible niche as a **rare specialist**, not a replacement for v12's deterministic portfolio.

## Deployment implications

The full audit is too expensive to run naively inside every live agent session: some schemas make repeated one-hot logistic fits slow. A practical v13 should:

1. keep the v12 quick and portfolio stages unchanged;
2. use Gemini only to propose one allowlisted family;
3. run a cheaper two-seed/two-auditor pre-gate;
4. require cross-model sign agreement;
5. compare the final planned model directly with the best portfolio OOF candidate; and
6. submit nothing when any gate fails.

The binary target is never transformed for normality. All transformations apply only to predictors and are justified by skew, missingness, cardinality, or predictor structure.